In [51]:
import numpy as np
import pandas as pd
import time
import logging
import matplotlib.pyplot as plt
from typing import List, Dict
from dataclasses import dataclass, field
from datetime import datetime
from datasets import load_dataset

In [ ]:
!pip install langchain langchain-community chromadb
!pip install datasets langchain chromadb sentence-transformers rank-bm25 pymupdf

In [52]:
# Configuration
CONFIG = {
    "embedding_model": "sentence-transformers/all-MiniLM-L6-v2",
    "dataset_name": "MarkrAI/AutoRAG-evaluation-2024-LLM-paper-v1",
    "dense_weight": 0.7,  # RRF Weight for Semantic search
    "top_k": 5}

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
logger = logging.getLogger(__name__)

### Data Loading

In [ ]:
def load_markrai_dataset():
    """Loads the 8.5k research corpus nodes and 520 QA test cases."""
    logger.info("Fetching corpus and QA subsets from Hugging Face...")
    corpus_ds = load_dataset(CONFIG['dataset_name'], "corpus", split="train")
    qa_ds = load_dataset(CONFIG['dataset_name'], "qa", split="train")
    
    docs = [{"content": r["contents"], "id": r["doc_id"]} for r in corpus_ds]
    test_set = [{"question": r["query"], "ground_truth": r["generation_gt"][0]} for r in qa_ds]
    
    return docs, test_set


### Hybrid Search Engine Architecture

In [ ]:
class HybridSearchEngine:
    """Combines Dense (Embeddings) and Sparse (BM25) for precision."""
    def __init__(self, documents: List[Dict]):
        from sentence_transformers import SentenceTransformer
        from rank_bm25 import BM25Okapi
        
        self.docs = documents
        self.model = SentenceTransformer(CONFIG["embedding_model"])
        self.embeddings = self.model.encode([d["content"] for d in documents], normalize_embeddings=True)
        
        tokenized_corpus = [d["content"].lower().split() for d in documents]
        self.bm25 = BM25Okapi(tokenized_corpus)

    def search(self, query: str, top_k: int = 5) -> List[Dict]:
        # Dense Rank
        q_emb = self.model.encode([query], normalize_embeddings=True)[0]
        dense_ranks = np.argsort(-np.dot(self.embeddings, q_emb))
        
        # Sparse Rank
        bm25_ranks = np.argsort(-self.bm25.get_scores(query.lower().split()))
        
        # Reciprocal Rank Fusion (RRF)
        k = 60
        scores = np.zeros(len(self.docs))
        for rank, idx in enumerate(dense_ranks):
            scores[idx] += CONFIG["dense_weight"] * (1 / (rank + k))
        for rank, idx in enumerate(bm25_ranks):
            scores[idx] += (1 - CONFIG["dense_weight"]) * (1 / (rank + k))
            
        return [dict(self.docs[i], rrf_score=scores[i]) for i in np.argsort(-scores)[:top_k]]


### RAG Pipeline and Monitoring

In [ ]:
@dataclass
class RAGResponse:
    query: str
    latency_ms: float
    recall_score: float

class RAGPipeline:
    def __init__(self, engine: HybridSearchEngine):
        self.engine = engine

    def query(self, user_query: str, ground_truth: str) -> RAGResponse:
        t0 = time.time()
        results = self.engine.search(user_query, top_k=CONFIG["top_k"])
        latency = (time.time() - t0) * 1000
        
        # Calculate Context Recall: Does retrieved text contain key GT words?
        context = " ".join([r["content"] for r in results]).lower()
        gt_tokens = set(ground_truth.lower().split())
        recall = sum(1 for t in gt_tokens if t in context) / len(gt_tokens) if gt_tokens else 0
        
        return RAGResponse(query=user_query, latency_ms=latency, recall_score=recall)


### Dashboard and Analysis

In [ ]:
def build_monitoring_dashboard(responses: List[RAGResponse]):
    """Generates the FAANG-style performance table and latency summary."""
    df = pd.DataFrame([{
        "Query": r.query[:50] + "...",
        "Recall": f"{r.recall_score:.1%}",
        "Latency (ms)": f"{r.latency_ms:.1f}"
    } for r in responses])
    
    avg_lat = np.mean([r.latency_ms for r in responses])
    avg_recall = np.mean([r.recall_score for r in responses])

    print("\n" + "="*85)
    print(f"{'MARKRAI RAG PERFORMANCE DASHBOARD':^85}")
    print("="*85)
    print(df.to_string(index=False))
    print("-" * 85)
    print(f"AVG LATENCY: {avg_lat:.2f} ms | AVG CONTEXT RECALL: {avg_recall:.2%}")
    print("="*85)

def main():
    docs, tests = load_markrai_dataset()
    engine = HybridSearchEngine(docs)
    pipeline = RAGPipeline(engine)
    
    # Run evaluation on first 10 samples
    results = [pipeline.query(t["question"], t["ground_truth"]) for t in tests[:10]]
    build_monitoring_dashboard(results)

if __name__ == "__main__":
    main()

2026-03-04 17:04:42,901 INFO Fetching corpus and QA subsets from Hugging Face...
2026-03-04 17:04:43,147 INFO HTTP Request: HEAD https://huggingface.co/datasets/MarkrAI/AutoRAG-evaluation-2024-LLM-paper-v1/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
2026-03-04 17:04:43,155 INFO HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/MarkrAI/AutoRAG-evaluation-2024-LLM-paper-v1/fc10b797869aaa4f8c7f007082d962c507b16fac/README.md "HTTP/1.1 200 OK"
2026-03-04 17:04:43,386 INFO HTTP Request: HEAD https://huggingface.co/datasets/MarkrAI/AutoRAG-evaluation-2024-LLM-paper-v1/resolve/fc10b797869aaa4f8c7f007082d962c507b16fac/AutoRAG-evaluation-2024-LLM-paper-v1.py "HTTP/1.1 404 Not Found"
2026-03-04 17:04:44,138 INFO HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/MarkrAI/AutoRAG-evaluation-2024-LLM-paper-v1/MarkrAI/AutoRAG-evaluation-2024-LLM-paper-v1.py "HTTP/1.1 404 Not Found"
2026-03-04 17:04:44,413 INFO HTTP Request: HEAD https

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
2026-03-04 17:04:50,251 INFO HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-03-04 17:04:50,261 INFO HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/config.json "HTTP/1.1 200 OK"
2026-03-04 17:04:50,558 INFO HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-03-04 17:04:50,568 INFO HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-

Batches:   0%|          | 0/268 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]


                          MARKRAI RAG PERFORMANCE DASHBOARD                          
                                                Query Recall Latency (ms)
Does the Turing Test assess a machine's ability to...   0.0%        135.4
Is the oa_temp or the zone_occ the most impactful ...  67.9%        112.0
What is the performance percentage increase observ...  88.9%         85.3
What are essential components of evaluating large ...  18.8%         74.9
How many errors were there in the Inference phase ...  62.5%        114.5
What is the meaning of PLP-former in the context o...  47.4%         87.6
How does the Qmsum value vary between the first an...   7.4%        101.5
What is the purpose of using the torch.einsum func...  45.5%        107.3
What is the result of applying the PAL attack agai...  88.9%         76.8
What is the process to deceive the test function a...  36.6%         88.0
-------------------------------------------------------------------------------------
AVG LATENCY: 